# Feature Engineering

This notebook performs following tasks-
- Raw data is loaded from an Athena table.
- Feature engineering techniques are applied—including binary label encoding, feature creation , and scaling.
-  Random Forest classifier is used to select top 10 features.
-  Final engineered dataset is saved as a CSV and uploaded to an S3 bucket.
-  A new Athena table is registered to enable querying this refined dataset.


## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import boto3
from pyathena import connect
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import sys
import os
import io

In [2]:
# Read the config
sys.path.append('../config')
import config
bucket = config.S3_BUCKET
s3_staging = config.ATHENA_STAGING
s3_engineered_path = f's3://{bucket}/final_project/engineered/'
engineered_filename = 'engineered_features.csv'

#print("Using bucket:", bucket, "Table:", s3_staging)

In [3]:
# Setup AWS session
session = boto3.session.Session()
region = session.region_name
s3_client = session.client('s3', region_name=region)

In [27]:
# List files in the prefix
prefix = 'final_project/staging/'
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
'''
# Display
if 'Contents' in response:
    print(f"📂 Objects under s3://{bucket}/{s3_staging}:")
    for obj in response['Contents']:
        print(" -", obj['Key'])
else:
    print("No objects found.")'''

'\n# Display\nif \'Contents\' in response:\n    print(f"📂 Objects under s3://{bucket}/{s3_staging}:")\n    for obj in response[\'Contents\']:\n        print(" -", obj[\'Key\'])\nelse:\n    print("No objects found.")'

# Feature engineering

In [5]:
# Connect to Athena and read raw table
db = 'final_project'
table = 'ddos_data'
conn = connect(region_name=region, s3_staging_dir=s3_staging)
query = f"SELECT * FROM {db}.{table}"
df = pd.read_sql(query, conn)

/tmp/ipykernel_1747/597086735.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [6]:
df.head()

,protocol,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [7]:
print(df.shape, df.columns, df.dtypes)

(221264, 78) Index(['protocol', 'flow_duration', 'total_fwd_packets',
       'total_backward_packets', 'fwd_packets_length_total',
       'bwd_packets_length_total', 'fwd_packet_length_max',
       'fwd_packet_length_min', 'fwd_packet_length_mean',
       'fwd_packet_length_std', 'bwd_packet_length_max',
       'bwd_packet_length_min', 'bwd_packet_length_mean',
       'bwd_packet_length_std', 'flow_bytes_per_s', 'flow_packets_per_s',
       'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max',
       'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std',
       'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags',
       'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length',
       'bwd_header_length', 'fwd_packets_per_s', 'bwd_packets_per_s',
       'packet_length_min', 'packet_length_max', 'packet_length_mean',
       'packet_length_std', 'packet_length_variance', 'fin_flag_count',
    

In [8]:
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 1])

flow_bytes_per_s        221264
flow_packets_per_s      221264
fwd_packets_per_s       221264
bwd_packets_per_s       221264
down_up_ratio           221264
fwd_avg_bytes_bulk      221264
fwd_avg_packets_bulk    221264
bwd_avg_bytes_bulk      221264
bwd_avg_packets_bulk    221264
dtype: int64


In [9]:
df.shape

(221264, 78)

In [10]:
# Drop columns with all NaNs
df_cleaned = df.dropna(axis=1, thresh=1)

In [11]:
df_cleaned.shape

(221264, 69)

In [12]:
# Binary label encoding
df_cleaned['label'] = df_cleaned['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)

/tmp/ipykernel_1747/2488267272.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['label'] = df_cleaned['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)


In [13]:
print(df_cleaned.head())
print(df_cleaned.shape, df_cleaned.columns, df_cleaned.dtypes)

   protocol  flow_duration  total_fwd_packets  total_backward_packets  \
0         6              3                  2                       0   
1         6            109                  1                       1   
2         6             52                  1                       1   
3         6             34                  1                       1   
4         6              3                  2                       0   

   fwd_packets_length_total  bwd_packets_length_total  fwd_packet_length_max  \
0                        12                         0                      6   
1                         6                         6                      6   
2                         6                         6                      6   
3                         6                         6                      6   
4                        12                         0                      6   

   fwd_packet_length_min  fwd_packet_length_mean  fwd_packet_length_std  ...  \


In [14]:
# Normalize numeric features
features = df.drop('label', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
X_scaled_df = pd.DataFrame(X_scaled, columns=features.columns)
X_scaled_df['label'] = df['label']

/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [15]:
# Feature selection using Random Forest
X = X_scaled_df.drop('label', axis=1)
y = X_scaled_df['label']
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

RandomForestClassifier(random_state=42)

In [16]:
# Select top 10 features
importances = clf.feature_importances_
top_features = X.columns[np.argsort(importances)[::-1][:10]]
df_final = X_scaled_df[top_features.tolist() + ['label']]

In [17]:
print(top_features)

Index(['fwd_packet_length_mean', 'subflow_fwd_bytes', 'fwd_packet_length_max',
       'fwd_packets_length_total', 'fwd_act_data_packets',
       'avg_fwd_segment_size', 'fwd_iat_std', 'init_fwd_win_bytes',
       'subflow_fwd_packets', 'fwd_header_length'],
      dtype='object')


In [18]:
# Convert DataFrame to CSV in memory
csv_buffer = io.StringIO()
df_final.to_csv(csv_buffer, index=False)

# Upload to S3
s3_client.put_object(
    Bucket=bucket,
    Key=f'final_project/engineered/{engineered_filename}',
    Body=csv_buffer.getvalue()
)

{'ResponseMetadata': {'RequestId': '2YR3QS0X031J1DED',
  'HostId': 'KpiXoembUWnbVVFEcz3AgBMMzCPjVnehpvI62AIEDBxGIOE+ti3GfAgHVfM4iC2qV9wm0texyWg=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'KpiXoembUWnbVVFEcz3AgBMMzCPjVnehpvI62AIEDBxGIOE+ti3GfAgHVfM4iC2qV9wm0texyWg=',
   'x-amz-request-id': '2YR3QS0X031J1DED',
   'date': 'Mon, 26 May 2025 20:16:45 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"274548232753c0067052798d2f6cff95"',
   'x-amz-checksum-crc32': 'cJ6dhA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"274548232753c0067052798d2f6cff95"',
 'ChecksumCRC32': 'cJ6dhA==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [31]:
# Append feature output path to config.py
with open('../config/config.py', 'a') as f:
    f.write(f"FEATURE_ENGINEERED_PATH = 's3://{bucket}/final_project/engineered/{engineered_filename}'\n")    

In [20]:
prefix = 'final_project/engineered/'
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)

In [29]:
# Filter for .csv files
'''csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

if csv_files:
    print("Found CSV files:")
    for file in csv_files:
        print("-", file)
else:
    print("No CSV files found.")'''

'csv_files = [obj[\'Key\'] for obj in response.get(\'Contents\', []) if obj[\'Key\'].endswith(\'.csv\')]\n\nif csv_files:\n    print("Found CSV files:")\n    for file in csv_files:\n        print("-", file)\nelse:\n    print("No CSV files found.")'

In [22]:
# Test 

# Choose first CSV file from list
key = csv_files[0]

# Load it into memory
obj = s3_client.get_object(Bucket=bucket, Key=key)
df_features = pd.read_csv(io.BytesIO(obj['Body'].read()))

# Preview it
print(f"📄 Preview of {key}")
display(df_features.head())


📄 Preview of final_project/engineered/engineered_features.csv


,fwd_packet_length_mean,subflow_fwd_bytes,fwd_packet_length_max,fwd_packets_length_total,fwd_act_data_packets,avg_fwd_segment_size,fwd_iat_std,init_fwd_win_bytes,subflow_fwd_packets,fwd_header_length,label
0,-0.317183,-0.288168,-0.28843,-0.288168,-0.190377,-0.317183,-0.487642,-0.530949,-0.188230,-0.191699,Benign
1,-0.317183,-0.289997,-0.28843,-0.289997,-0.271105,-0.317183,-0.487642,-0.531447,-0.252449,-0.244410,Benign
2,-0.317183,-0.289997,-0.28843,-0.289997,-0.271105,-0.317183,-0.487642,-0.531447,-0.252449,-0.244410,Benign
3,-0.317183,-0.289997,-0.28843,-0.289997,-0.271105,-0.317183,-0.487642,-0.531198,-0.252449,-0.244410,Benign
4,-0.317183,-0.288168,-0.28843,-0.288168,-0.190377,-0.317183,-0.487642,-0.531073,-0.188230,-0.191699,Benign


In [23]:
print(df_features.head())
print(df_features.shape)
print(df_features.columns)
print(df_features.dtypes)

   fwd_packet_length_mean  subflow_fwd_bytes  fwd_packet_length_max  \
0               -0.317183          -0.288168               -0.28843   
1               -0.317183          -0.289997               -0.28843   
2               -0.317183          -0.289997               -0.28843   
3               -0.317183          -0.289997               -0.28843   
4               -0.317183          -0.288168               -0.28843   

   fwd_packets_length_total  fwd_act_data_packets  avg_fwd_segment_size  \
0                 -0.288168             -0.190377             -0.317183   
1                 -0.289997             -0.271105             -0.317183   
2                 -0.289997             -0.271105             -0.317183   
3                 -0.289997             -0.271105             -0.317183   
4                 -0.288168             -0.190377             -0.317183   

   fwd_iat_std  init_fwd_win_bytes  subflow_fwd_packets  fwd_header_length  \
0    -0.487642           -0.530949          

In [24]:
df_features.columns

Index(['fwd_packet_length_mean', 'subflow_fwd_bytes', 'fwd_packet_length_max',
       'fwd_packets_length_total', 'fwd_act_data_packets',
       'avg_fwd_segment_size', 'fwd_iat_std', 'init_fwd_win_bytes',
       'subflow_fwd_packets', 'fwd_header_length', 'label'],
      dtype='object')

In [25]:
print(X_scaled_df[df_features.columns].head())
print(X_scaled_df.shape)

print(X_scaled_df.dtypes)

   fwd_packet_length_mean  subflow_fwd_bytes  fwd_packet_length_max  \
0               -0.317183          -0.288168               -0.28843   
1               -0.317183          -0.289997               -0.28843   
2               -0.317183          -0.289997               -0.28843   
3               -0.317183          -0.289997               -0.28843   
4               -0.317183          -0.288168               -0.28843   

   fwd_packets_length_total  fwd_act_data_packets  avg_fwd_segment_size  \
0                 -0.288168             -0.190377             -0.317183   
1                 -0.289997             -0.271105             -0.317183   
2                 -0.289997             -0.271105             -0.317183   
3                 -0.289997             -0.271105             -0.317183   
4                 -0.288168             -0.190377             -0.317183   

   fwd_iat_std  init_fwd_win_bytes  subflow_fwd_packets  fwd_header_length  \
0    -0.487642           -0.530949          

In [34]:
print("Feature Engineered file:", engineered_filename, "Stored.")

Feature Engineered file: engineered_features.csv Stored.


In [26]:
print(f"Feature-engineering completed.")

Feature-engineering completed.
